## Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, GridSearchCV, train_test_split
from sklearn.linear_model import LassoCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import make_scorer
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import dask.dataframe as dd
import lightgbm as lgb
import optuna
# import xgboost as xgb
# import catboost as cb
import seaborn as sns
from scipy.stats import uniform, randint

## Model Building and Data Handling

### Define scoring using myscore

In [2]:
def weighted_mae_calculaion(y_true, y_pred, weights):
    errors = np.abs(y_true - y_pred) * weights
    weighted_errors = errors*weights
    return np.sum(weighted_errors) / np.sum(weights)
def weighted_mae(y_true, y_pred, weights):
    y_true = y_true.merge(weights, on = 'unique_id', how = 'left')
    weights = y_true['weight'].values
    y_true = y_true['sales'].values
    return weighted_mae_calculaion(y_true, y_pred, weights)

### Get weight

In [3]:
# read from "test_weights.csv" using read.csv
weights = pd.read_csv("test_weights.csv")

## Model Training

### Import Data

In [12]:
file_path = "not_encoded_sales_train.csv"
test_file_path = "not_encoded_sales_test.csv"

### Train Model

In [13]:
chunk_size = 1000000  # Adjust based on your memory capacity
train_ratio = 0.8
total_rows = sum(1 for _ in open("processed_sales_train2.csv")) - 1
total_chunks = total_rows // chunk_size
train_chunks = int(total_chunks * train_ratio)

In [40]:
def objective(trial):
    try:
        param = {
            "objective": "regression",
            "metric": "rmse",
            "boosting_type": "gbdt",
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 20, 50),  # Reduced range
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 100, 500),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.01, 0.1),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.7, 0.9),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.7, 0.9),
            "bagging_freq": 5,
            "max_bin": 255,
            "verbosity": -1,
            "random_state": 42
        }
        model = lgb.LGBMRegressor(**param)
        valid_preds = []
        valid_y = []

        for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):  # Adjust chunk size as needed
            X_chunk = chunk.drop(columns = ['sales'])
            X_chunk['warehouse'] = X_chunk['warehouse'].astype('category')
            X_chunk['holiday_name'] = X_chunk['holiday_name'].astype('category')
            y_chunk = chunk['sales']
            train_data = lgb.Dataset(X_chunk, label = y_chunk, categorical_feature = ['warehouse', 'holiday_name'], free_raw_data = False)  # Adjust based on your data structure
            if i < train_chunks:
                if model is None:
                    model = lgb.train(param, train_data, num_boost_round = 100)
                else:
                     model = lgb.train(param, train_data, num_boost_round = 100, init_model = model, keep_training_booster = True)
            else:
                 valid_preds = model.predict(X_chunk)
                 valid_y.extend(y_chunk)
        wmae = weighted_mae(chunk[['unique_id','sales']], valid_preds, weights)   
        return wmae
    except Exception as e:
         print(f"Trial {trial.number} failed due to error: {str(e)}")
         return float("inf")

In [39]:
# sales_train, sales_test = PCA_transformer(sales_train, sales_test)
# sales_train = lgb.Dataset(sales_train.drop(columns = ['sales']), sales_train['sales'])
# sales_train.save_binary("sales_train.bin")
# sales_train = lgb.Dataset("sales_train.bin")
pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
study = optuna.create_study(direction="minimize", pruner=pruner)
study.optimize(objective, n_trials = 2)
print("Best trial: score {},\nparams {}".format(study.best_trial.value, study.best_trial.params))

[I 2025-02-13 17:53:38,608] A new study created in memory with name: no-name-7413b317-3f59-423d-a655-9732460c8f8c
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
[I 2025-02-13 17:53:38,656] Trial 0 finished with value: inf and parameters: {'n_estimators': 129, 'learning_rate': 0.05583685530999711, 'num_leaves': 39, 'max_depth': 11, 'min_data_in_leaf': 254, 'min_gain_to_split': 0.04215272945295401, 'feature_fraction': 0.829468085859739, 'bagging_fraction': 0.7014619586359592}. Best is trial 0 with value: inf.
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
[I 2025-02-13 17:53:38,708] Trial 1 finished with value: inf and parameter

Trial 0 failed due to error: The entry associated with the validation name "valid_0" and the metric name "mae" is not found in the evaluation result list [].
Trial 1 failed due to error: The entry associated with the validation name "valid_0" and the metric name "mae" is not found in the evaluation result list [].
Best trial: score inf,
params {'n_estimators': 129, 'learning_rate': 0.05583685530999711, 'num_leaves': 39, 'max_depth': 11, 'min_data_in_leaf': 254, 'min_gain_to_split': 0.04215272945295401, 'feature_fraction': 0.829468085859739, 'bagging_fraction': 0.7014619586359592}


### Load the model and Make Prediction

In [33]:
best_params = study.best_params
model = None

for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):
    X_chunk = chunk.drop(columns = ['sales'])
    X_chunk['warehouse'] = X_chunk['warehouse'].astype('category')
    X_chunk['holiday_name'] = X_chunk['holiday_name'].astype('category')
    y_chunk = chunk['sales']
    train_data = lgb.Dataset(X_chunk, label = y_chunk, categorical_feature = ['warehouse', 'holiday_name'], free_raw_data = False)
    if i< train_chunks:
        if model is None:
            model = lgb.train(best_params, train_data)
        else:
             model = lgb.train(best_params, train_data, init_model = model, keep_training_booster = True)
model.save_model("final_lightgbm_model.txt")

c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


In [34]:
chunk_size = 10000
predictions = []
for chunk in pd.read_csv(test_file_path, chunksize=chunk_size):
    chunk[['warehouse', 'holiday_name']] = chunk[['warehouse', 'holiday_name']].astype('category')
    X_test = chunk
    preds = model.predict(X_test)
    predictions.extend(preds)
pred_df = pd.DataFrame({'predictions': predictions})
pred_df.to_csv("final_predictions.csv", index=False)